In [ ]:
from IPython.display import display, HTML
display(HTML("<style>table {margin-left: 0 !important;} .MathJax_Display, .MathJax {text-align: left !important;}</style>"))

# Week 4 Lab — Supervised Learning: Regression
**Introduction to Data Science — DBAS 5015**

---

<!-- ============================================================
  WRITING STYLE — apply to all markdown cells in this notebook
  - Direct and to the point. Say what the concept is and move on.
  - No hedging. State things plainly.
  - Active voice. 'The function returns a value' not 'A value is returned.'
  - Short sentences for instructions. Students need to scan them quickly.
  - Exercise instructions must be unambiguous — one reading, one meaning.
  - No filler. Every sentence should add information or cut it.
  ============================================================ -->

### What This Lab Covers
- Building and fitting a `LinearRegression` model with scikit-learn
- Evaluating with MAE, RMSE, and R²
- Interpreting model coefficients
- Plotting and reading residuals

**Estimated time:** 75–90 minutes

---

## The Dataset

This lab uses a synthetic property sales dataset. Each row is a residential property sale. The goal is to predict `sale_price` — a continuous numeric target — from property characteristics. Run the cell below to generate the dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

np.random.seed(42)
n = 300

neighbourhoods = np.random.choice(
    ['Downtown', 'Midtown', 'Suburbs', 'Rural'], n,
    p=[0.20, 0.30, 0.35, 0.15]
)
property_type = np.random.choice(
    ['Condo', 'Townhouse', 'Detached'], n,
    p=[0.40, 0.35, 0.25]
)
sqft       = np.random.randint(400, 3500, n)
bedrooms   = np.random.randint(1, 6, n)
bathrooms  = np.random.choice([1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0], n)
age_years  = np.random.randint(0, 50, n)
has_garage = np.random.choice([0, 1], n, p=[0.35, 0.65])

neigh_mult = np.where(neighbourhoods == 'Downtown', 1.40,
             np.where(neighbourhoods == 'Midtown',  1.20,
             np.where(neighbourhoods == 'Suburbs',  1.00, 0.75)))

type_premium = np.where(property_type == 'Detached',  50000,
               np.where(property_type == 'Townhouse', 20000, 0))

base_price = (
    sqft      * 250 +
    bedrooms  * 15000 +
    bathrooms * 8000 -
    age_years * 1500 +
    has_garage * 25000
)

sale_price = np.round(
    (base_price * neigh_mult + type_premium)
    + np.random.normal(0, 30000, n),
    -2
)
sale_price = np.maximum(sale_price, 80000)

df = pd.DataFrame({
    'neighbourhood': neighbourhoods,
    'property_type': property_type,
    'sqft':          sqft,
    'bedrooms':      bedrooms,
    'bathrooms':     bathrooms,
    'age_years':     age_years,
    'has_garage':    has_garage,
    'sale_price':    sale_price,
})

print(f'Dataset ready: {df.shape[0]} rows, {df.shape[1]} columns')
print(df.dtypes)

---
## Part 1 — Explore the Dataset

Before fitting any model, understand the target and its relationship to the features. Regression models perform better — and their outputs are easier to interpret — when you know the shape of the data.

In [ ]:
# Target distribution
print('Sale price summary:')
print(df['sale_price'].describe().round(0))

# Distribution plot
plt.figure(figsize=(8, 3))
plt.hist(df['sale_price'], bins=30, edgecolor='white')
plt.xlabel('Sale Price ($)')
plt.ylabel('Count')
plt.title('Distribution of Sale Price')
plt.tight_layout()
plt.show()

---
### 💼 Business Context — Know Your Target Before Modelling

A skewed target distribution does not automatically mean you need to transform it — but it does mean your error metrics will be dominated by extreme values. A right-skewed price distribution (many modest homes, few luxury properties) means RMSE will be pulled up by the luxury end. MAE will give a more representative picture of typical prediction error for the average property. Report both and note the difference.

---

### ✏️ Exercise 1.1 — Quick EDA

1. Print the mean sale price for each `neighbourhood` using `groupby`.
2. Print the mean sale price for each `property_type`.
3. Compute the correlation between `sqft` and `sale_price`. Print the value and write a one-sentence interpretation.
4. Separate the dataset into `X` (all columns except `sale_price`) and `y` (`sale_price`).

In [ ]:
# Exercise 1.1

# 1. Mean sale price by neighbourhood
neigh_means = 
print('Mean sale price by neighbourhood:')
print(neigh_means)

# 2. Mean sale price by property type
type_means = 
print('\nMean sale price by property type:')
print(type_means)

# 3. Correlation between sqft and sale_price
corr = 
print(f'\nCorrelation — sqft vs sale_price: {corr:.3f}')
# Interpretation:
# 

# 4. Separate features and target
X = 
y = 
print(f'\nX shape: {X.shape}, y shape: {y.shape}')

---
## Part 2 — Preprocess and Split

This is the same preprocessing pipeline from Week 3: encode categorical columns, scale numeric columns, split before fitting. Nothing changes except the dataset.

In [ ]:
# Column lists
categorical_cols = ['neighbourhood', 'property_type']
numerical_cols   = ['sqft', 'bedrooms', 'bathrooms', 'age_years', 'has_garage']

# Build the preprocessor
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False), categorical_cols),
    ('num', StandardScaler(), numerical_cols),
])

# Split — before fitting anything
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fit on training data only, transform both sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f'Training set: {X_train_processed.shape}')
print(f'Test set:     {X_test_processed.shape}')
print(f'\nFeature names:')
print(preprocessor.get_feature_names_out())

---
### 🤖 ML Connection — Regression Targets Do Not Need Stratification

`stratify=y` is used in classification to preserve class proportions in both splits. Regression targets are continuous — there are no classes to preserve. A random split is sufficient. For heavily skewed targets, some practitioners bin the target temporarily for stratification, but this is rarely necessary and not covered here.

---

### ✏️ Exercise 2.1 — Verify the Split

1. Print the mean and standard deviation of `y_train` and `y_test`.
2. Confirm they are similar — a large difference would indicate the split produced unrepresentative sets.
3. Print the shape of `X_train_processed` and name how many columns came from encoding vs. scaling.

In [ ]:
# Exercise 2.1

# 1. Mean and std of y_train and y_test
print(f'y_train — mean: ${y_train.mean():,.0f}  std: ${y_train.std():,.0f}')
print(f'y_test  — mean: ${y_test.mean():,.0f}  std: ${y_test.std():,.0f}')

# 2. Are they similar? Write a comment.
# 

# 3. Shape and column breakdown
print(f'\nX_train_processed shape: {X_train_processed.shape}')
# How many columns came from encoding?
ohe_count = 
# How many from scaling?
num_count = 
print(f'Encoded columns: {ohe_count}  |  Scaled columns: {num_count}')

---
## Part 3 — Fit and Evaluate the Model

With preprocessed data ready, fitting the model is two lines. Evaluation takes more attention — you need all three metrics on both train and test sets.

In [ ]:
# Fit the model
model = LinearRegression()
model.fit(X_train_processed, y_train)

# Predict on both sets
y_pred_train = model.predict(X_train_processed)
y_pred_test  = model.predict(X_test_processed)

# Evaluate on the test set
mae  = mean_absolute_error(y_test, y_pred_test)
rmse = root_mean_squared_error(y_test, y_pred_test)
r2   = r2_score(y_test, y_pred_test)

print('Test set metrics:')
print(f'  MAE:  ${mae:,.0f}')
print(f'  RMSE: ${rmse:,.0f}')
print(f'  R²:   {r2:.3f}')

---
### 🤖 ML Connection — RMSE vs. MAE: Why Both Matter

MAE is the average error — every error counts equally. RMSE penalizes large errors more severely because it squares them before averaging. The ratio RMSE/MAE tells you something about the error distribution: if RMSE is much larger than MAE, the model has a few very large errors. For a property valuation model, a $300,000 error is not just '10x worse' than a $30,000 error — it may make the model unusable for the properties it gets most wrong. In those cases, RMSE is the metric to minimize.

---

### ✏️ Exercise 3.1 — Full Metric Comparison

1. Compute MAE, RMSE, and R² on **both** the training set and the test set.
2. Print all six values in a clear format.
3. Compare train and test R². Write a comment interpreting what the gap (or lack of gap) tells you.
4. Compute the ratio RMSE / MAE on the test set. Write a comment on what this ratio reveals.

In [ ]:
# Exercise 3.1

# 1–2. Metrics on train and test
train_mae  = mean_absolute_error(y_train, y_pred_train)
train_rmse = 
train_r2   = 

test_mae   = mean_absolute_error(y_test, y_pred_test)  # provided — do not change
test_rmse  = 
test_r2    = 

print(f'{"Metric":<8}  {"Train":>12}  {"Test":>12}')
print(f'{"MAE":<8}  ${train_mae:>11,.0f}  ${test_mae:>11,.0f}')
print(f'{"RMSE":<8}  ${train_rmse:>11,.0f}  ${test_rmse:>11,.0f}')
print(f'{"R²":<8}  {train_r2:>12.3f}  {test_r2:>12.3f}')

# 3. Train vs. test R² interpretation
# 

# 4. RMSE / MAE ratio on test set
ratio = 
print(f'\nRMSE / MAE ratio: {ratio:.2f}')
# What does this reveal?
# 

---
## Part 4 — Interpreting Coefficients

The model has learned a coefficient for each feature. These coefficients tell you which features drive predicted price up or down — and by how much (in units of standard deviations, since the data is scaled).

In [ ]:
# Build a coefficient table
feature_names = preprocessor.get_feature_names_out()

coef_df = pd.DataFrame({
    'feature':     feature_names,
    'coefficient': model.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print('Coefficients (sorted by absolute value):')
print(coef_df.to_string(index=False))
print(f'\nIntercept: ${model.intercept_:,.0f}')

---
### 💼 Business Context — Communicating Model Results to Stakeholders

Stakeholders do not read coefficient tables. They ask: 'What drives price?' Translate the table into plain language: 'Square footage is the strongest single predictor — a one-standard-deviation increase in size adds roughly $X to the predicted price. Location matters almost as much — a Downtown property commands a significant premium over the baseline.' The coefficient table is your evidence; the narrative is what you present.

---

### ✏️ Exercise 4.1 — Interpret and Visualize Coefficients

1. Identify the three features with the largest positive coefficients and the three with the largest negative coefficients.
2. Create a horizontal bar chart of all coefficients, sorted by value. Color positive bars one color and negative bars another.
3. Write a 3–4 sentence business interpretation of the top three drivers of sale price.

In [ ]:
# Exercise 4.1

# 1. Top 3 positive and top 3 negative
top_positive = coef_df.nlargest(3, 'coefficient')
top_negative = coef_df.nsmallest(3, 'coefficient')
print('Top 3 positive coefficients:')
print(top_positive.to_string(index=False))
print('\nTop 3 negative coefficients:')
print(top_negative.to_string(index=False))

# 2. Horizontal bar chart
sorted_coef = coef_df.sort_values('coefficient')
colors = ['#d73027' if c < 0 else '#4575b4' for c in sorted_coef['coefficient']]

plt.figure(figsize=(9, 5))
plt.barh(sorted_coef['feature'], sorted_coef['coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coefficient (scaled units)')
plt.title('Linear Regression Coefficients — Property Price Model')
plt.tight_layout()
plt.show()

# 3. Business interpretation (fill in after viewing the chart)
# 
# 
# 
# 

---
## Challenge Exercise

This section is optional — attempt it if you finish early or want a stretch.

### Residual Analysis

A clean coefficient table and a good R² do not guarantee the model is well-specified. Residual plots reveal patterns the metrics miss.

1. Compute residuals on the test set: `residuals = y_test - y_pred_test`.
2. Create a scatter plot of `y_pred_test` (x-axis) vs. `residuals` (y-axis). Add a horizontal line at y = 0.
3. Create a histogram of the residuals with 30 bins.
4. Examine both plots and answer:
   - Is the scatter random around zero, or is there a pattern?
   - Are residuals roughly normally distributed?
   - Is there a funnel shape (heteroscedasticity)?
5. **Stretch:** Compute the mean residual for each neighbourhood by joining `residuals` back to the test set's neighbourhood column. If one neighbourhood has a consistently positive or negative mean residual, what does that tell you?

In [ ]:
# Challenge — residual analysis

# 1. Compute residuals
residuals = 

# 2. Scatter plot — predicted vs. residuals
plt.figure(figsize=(8, 4))
# your code here
plt.show()

# 3. Histogram of residuals
plt.figure(figsize=(6, 4))
# your code here
plt.show()

# 4. Interpretation
# Random scatter? 
# Approximately normal? 
# Funnel shape? 

# 5. Stretch — mean residual by neighbourhood
X_test_reset = X_test.reset_index(drop=True)
resid_by_neigh = 
print('Mean residual by neighbourhood:')
print(resid_by_neigh)
# Interpretation:
# 

---
## Week 4 Summary

| Concept | Key Takeaway |
|:--------|:-------------|
| Regression target | Continuous numeric value — predicts *how much*, not *which category* |
| `LinearRegression().fit()` | Learns coefficients by minimizing squared error on training data |
| Predict on both sets | Always generate predictions for train and test — you need both to detect overfitting |
| MAE | Average absolute error — same units as target, equal weight to all errors |
| RMSE | Square-root of mean squared error — penalizes large errors more than MAE |
| RMSE / MAE ratio | Close to 1.0 = consistent errors; much > 1.0 = a few large outlier errors |
| R² | Proportion of variance explained — compare train and test values to detect overfitting |
| `.coef_` | Learned coefficients — sorted by absolute value to find most influential features |
| Scaled coefficients | Directly comparable across features — largest absolute value = most influential |
| Residual plot | Random scatter = healthy; patterns = model limitation or missing feature |

---
**Next week:** Classification — predicting categories instead of numbers. You will use logistic regression and decision trees, and learn a new set of evaluation metrics: confusion matrix, precision, recall, and F1.